In [ ]:
from google.colab import files
from pathlib import Path

Path("/content/ddinter").mkdir(exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    open(f"/content/ddinter/{name}", "wb").write(data)

DATA_DIR = Path("/content/ddinter")
print(list(DATA_DIR.glob("*.csv")))

Saving ddinter_downloads_code_V.csv to ddinter_downloads_code_V (1).csv
Saving ddinter_downloads_code_R.csv to ddinter_downloads_code_R (1).csv
Saving ddinter_downloads_code_L.csv to ddinter_downloads_code_L (1).csv
Saving ddinter_downloads_code_P.csv to ddinter_downloads_code_P (1).csv
Saving ddinter_downloads_code_H.csv to ddinter_downloads_code_H (1).csv
Saving ddinter_downloads_code_D.csv to ddinter_downloads_code_D (1).csv
Saving ddinter_downloads_code_B.csv to ddinter_downloads_code_B (1).csv
Saving ddinter_downloads_code_A.csv to ddinter_downloads_code_A (1).csv
[PosixPath('/content/ddinter/ddinter_downloads_code_H (1).csv'), PosixPath('/content/ddinter/ddinter_downloads_code_P (1).csv'), PosixPath('/content/ddinter/ddinter_downloads_code_A.csv'), PosixPath('/content/ddinter/ddinter_downloads_code_D (1).csv'), PosixPath('/content/ddinter/ddinter_downloads_code_L.csv'), PosixPath('/content/ddinter/ddinter_downloads_code_R (1).csv'), PosixPath('/content/ddinter/ddinter_downloads_c

In [ ]:
files = sorted(f for f in DATA_DIR.glob("*.csv") if "(1)" not in f.name)
print(f"Found {len(files)} CSV files")
for f in files:
    print(" -", f.name)

frames = []
for f in files:
    d = pd.read_csv(f)
    d["source_file"] = f.name
    frames.append(d)

raw = pd.concat(frames, ignore_index=True)
print("\nColumns:", list(raw.columns))
raw.head()

Found 8 CSV files
 - ddinter_downloads_code_A.csv
 - ddinter_downloads_code_B.csv
 - ddinter_downloads_code_D.csv
 - ddinter_downloads_code_H.csv
 - ddinter_downloads_code_L.csv
 - ddinter_downloads_code_P.csv
 - ddinter_downloads_code_R.csv
 - ddinter_downloads_code_V.csv

Columns: ['DDInterID_A', 'Drug_A', 'DDInterID_B', 'Drug_B', 'Level', 'source_file']


,DDInterID_A,Drug_A,DDInterID_B,Drug_B,Level,source_file
0,DDInter1263,Naltrexone,DDInter1,Abacavir,Moderate,ddinter_downloads_code_A.csv
1,DDInter1,Abacavir,DDInter1348,Orlistat,Moderate,ddinter_downloads_code_A.csv
2,DDInter58,Aluminum hydroxide,DDInter582,Dolutegravir,Major,ddinter_downloads_code_A.csv
3,DDInter112,Aprepitant,DDInter582,Dolutegravir,Minor,ddinter_downloads_code_A.csv
4,DDInter138,Attapulgite,DDInter582,Dolutegravir,Major,ddinter_downloads_code_A.csv


In [ ]:
def find_col(df, patterns):
    for c in df.columns:
        if any(re.search(p, c, re.I) for p in patterns):
            return c
    return None

COL_A = find_col(raw, [r"^drug_?a$", r"drug.*a$"])
COL_B = find_col(raw, [r"^drug_?b$", r"drug.*b$"])
COL_LEVEL = find_col(raw, [r"level", r"severity"])

print("Drug A column:", COL_A)
print("Drug B column:", COL_B)
print("Severity column:", COL_LEVEL)
assert COL_A and COL_B and COL_LEVEL, "Set COL_A, COL_B, COL_LEVEL manually above."

Drug A column: Drug_A
Drug B column: Drug_B
Severity column: Level


In [ ]:
df = raw[[COL_A, COL_B, COL_LEVEL, "source_file"]].copy()
df.columns = ["drug_a", "drug_b", "severity", "source_file"]

# normalise names for counting (lowercase, trimmed, single spaces)
def norm(s):
    return re.sub(r"\s+", " ", str(s).strip().lower()) if pd.notna(s) else s

df["a"] = df["drug_a"].map(norm)
df["b"] = df["drug_b"].map(norm)

# unordered pair key: (A,B) and (B,A) are the same interaction
df["pair"] = df.apply(lambda r: tuple(sorted([r["a"], r["b"]])), axis=1)

medicines = set(df["a"].dropna()) | set(df["b"].dropna())

print(f"Rows (as stored):              {len(df)}")
print(f"Unique medicines:              {len(medicines)}")
print(f"Unique unordered pairs:        {df['pair'].nunique()}")
print("\nSeverity levels (rows):")
print(df["severity"].value_counts(dropna=False))
print("\nSeverity levels (unique pairs):")
print(df.drop_duplicates("pair")["severity"].value_counts(dropna=False))

Rows (as stored):              222383
Unique medicines:              1939
Unique unordered pairs:        160235

Severity levels (rows):
severity
Moderate    130367
Unknown      47182
Major        33896
Minor        10938
Name: count, dtype: int64

Severity levels (unique pairs):
severity
Moderate    96675
Unknown     29813
Major       26914
Minor        6833
Name: count, dtype: int64


In [ ]:
issues = {}

# missing values
issues["missing_values"] = int(df[["drug_a", "drug_b", "severity"]].isna().sum().sum())

# exact duplicate rows
issues["exact_duplicate_rows"] = int(df.duplicated(["a", "b", "severity"]).sum())

# same interaction stored in both orders (A,B) and (B,A)
ordered = df.drop_duplicates(["a", "b"])
issues["pairs_stored_in_both_orders"] = int(len(ordered) - ordered["pair"].nunique())

# same unordered pair with different severity
sev_per_pair = df.groupby("pair")["severity"].nunique()
conflicts = sev_per_pair[sev_per_pair > 1]
issues["pairs_with_conflicting_severity"] = int(len(conflicts))

# unknown / unlabeled severity
issues["unknown_severity_rows"] = int(df["severity"].astype(str).str.lower().isin(["unknown", "nan", ""]).sum())

# names that differ only by case/whitespace in the raw data
raw_names = pd.Series(list(set(df["drug_a"].dropna()) | set(df["drug_b"].dropna())))
variants = raw_names.groupby(raw_names.map(norm)).nunique()
issues["names_with_spelling_variants"] = int((variants > 1).sum())

# combination-like names (contain '+', '/', ' and ')
combo = [m for m in medicines if re.search(r"\+|/| and ", m)]
issues["combination_like_names"] = len(combo)

for k, v in issues.items():
    print(f"{k:35s} {v}")

if len(conflicts):
    print("\nExample conflicting pairs:")
    ex = df[df["pair"].isin(conflicts.index[:5])][["drug_a", "drug_b", "severity", "source_file"]]
    display(ex.sort_values(["drug_a", "drug_b"]))

missing_values                      0
exact_duplicate_rows                62148
pairs_stored_in_both_orders         0
pairs_with_conflicting_severity     0
unknown_severity_rows               47182
names_with_spelling_variants        0
combination_like_names              4


In [ ]:
labelled = df["severity"].notna() & ~df["severity"].astype(str).str.lower().isin(["unknown", ""])
print(f"Rows with a usable severity label: {labelled.sum()} of {len(df)} ({labelled.mean():.1%})")

Rows with a usable severity label: 175201 of 222383 (78.8%)


In [ ]:
SAMPLE = [
    "atorvastatin", "simvastatin", "rosuvastatin", "fluvastatin",
    "pantoprazole", "esomeprazole", "ranitidine", "amlodipine", "amiodarone",
    "verapamil", "warfarin", "clopidogrel", "sitagliptin", "linagliptin",
    "carvedilol", "loperamide", "metildigoxin", "risperidone",
    "spironolactone", "ciprofloxacin", "calcium carbonate",
]

if REGISTRY_CSV is not None and Path(REGISTRY_CSV).exists():
    reg = pd.read_csv(REGISTRY_CSV)
    substances = reg[REGISTRY_SUBSTANCE_COL].dropna().map(norm).unique().tolist()
    source_label = "BiH registry file"
else:
    substances = [norm(s) for s in SAMPLE]
    source_label = "SAMPLE list (registry not loaded)"

print("Matching source:", source_label, f"({len(substances)} substances)")

ddi_names = sorted(medicines)
exact, fuzzy, unmatched = [], [], []

for s in substances:
    if s in medicines:
        exact.append(s)
    else:
        close = difflib.get_close_matches(s, ddi_names, n=1, cutoff=0.85)
        if close:
            fuzzy.append((s, close[0]))
        else:
            unmatched.append(s)

print(f"\nExact matches:   {len(exact)}")
print(f"Fuzzy matches:   {len(fuzzy)}  (need manual check)")
print(f"Unmatched:       {len(unmatched)}")
print(f"Match rate (exact + fuzzy): {(len(exact)+len(fuzzy))/len(substances):.1%}")

if fuzzy:
    print("\nFuzzy candidates:")
    for s, m in fuzzy: print(f"  {s}  ->  {m}")
if unmatched:
    print("\nUnmatched (likely spelling/INN differences or not in DDInter):")
    for s in unmatched: print("  ", s)

Matching source: SAMPLE list (registry not loaded) (21 substances)

Exact matches:   20
Fuzzy matches:   0  (need manual check)
Unmatched:       1
Match rate (exact + fuzzy): 95.2%

Unmatched (likely spelling/INN differences or not in DDInter):
   metildigoxin


In [ ]:
print("=== Handout numbers ===")
print(f"Rows:                      {len(df)}")
print(f"Columns used:              drug_a, drug_b, severity (+ source file)")
print(f"Unique medicines:          {len(medicines)}")
print(f"Unique unordered pairs:    {df['pair'].nunique()}")
print(f"Severity levels:           {sorted(df['severity'].dropna().astype(str).unique())}")
print(f"Target usable:             {labelled.mean():.1%} of rows have a severity label")
print(f"Matching source:           {source_label}")
print(f"Matched / total:           {len(exact)+len(fuzzy)} / {len(substances)}")
print("Quality issues found:     ", {k: v for k, v in issues.items() if v})

=== Handout numbers ===
Rows:                      222383
Columns used:              drug_a, drug_b, severity (+ source file)
Unique medicines:          1939
Unique unordered pairs:    160235
Severity levels:           ['Major', 'Minor', 'Moderate', 'Unknown']
Target usable:             78.8% of rows have a severity label
Matching source:           SAMPLE list (registry not loaded)
Matched / total:           20 / 21
Quality issues found:      {'exact_duplicate_rows': 62148, 'unknown_severity_rows': 47182, 'combination_like_names': 4}


In [ ]:
checks = [("atorvastatin","amlodipine"), ("simvastatin","amlodipine"),
          ("simvastatin","amiodarone"), ("atorvastatin","pantoprazole"),
          ("warfarin","simvastatin"), ("clopidogrel","rosuvastatin")]
pairs = set(df["pair"])
for a, b in checks:
    print(f"{a} + {b}:", tuple(sorted([a, b])) in pairs)

atorvastatin + amlodipine: False
simvastatin + amlodipine: True
simvastatin + amiodarone: True
atorvastatin + pantoprazole: True
warfarin + simvastatin: True
clopidogrel + rosuvastatin: True


In [ ]:
print(df[(df["a"].isin(["atorvastatin","amlodipine"])) & (df["b"].isin(["atorvastatin","amlodipine"]))])
print("atorvastatin rows:", ((df["a"]=="atorvastatin")|(df["b"]=="atorvastatin")).sum())
print("amlodipine rows:", ((df["a"]=="amlodipine")|(df["b"]=="amlodipine")).sum())

Empty DataFrame
Columns: [drug_a, drug_b, severity, source_file, a, b, pair]
Index: []
atorvastatin rows: 350
amlodipine rows: 343


In [ ]:
print([m for m in medicines if "digox" in m])

['digoxin immune fab (ovine)', 'digoxin']
